## Complete and Intuitive Guide to the Histogram Filter

### Discrete State Estimation for Lane Keeping (Lane Filter in Duckietown)

Welcome to the third notebook in the series on probabilistic localization! In the previous notebooks, we explored two different approaches:

1. **The Kalman Filter (KF):** Describes uncertainty using a continuous mathematical formula (mean $\mu$ and covariance matrix $\Sigma$), but strictly requires the world to be linear and Gaussian.
2. **The Particle Filter (PF):** Uses random Monte Carlo sampling with free particles in continuous space.

### Why Do We Need the Histogram Filter?

In many real-world mobile robotics and autonomous driving applications, we do not need to compute complex integrals or sample particles randomly:

* We can **divide the state space into a fixed grid of cells (bins)**, just like a chessboard or a 2D histogram!
* Each cell in the grid represents a discrete hypothesis (its center of mass, or centroid).
* To each cell, we assign a **probability (or probability mass $w^i_t$)**, subject to the condition that the sum of all cells equals $1$:

$$\sum_{i=1}^N w_t^i = 1$$


* Mathematically, this discrete distribution is described as a weighted sum of Dirac delta functions:

$$bel(\mathbf{x}_t) = \sum_{i=1}^N w_t^i \delta(\mathbf{x}_t - \mathbf{x}^i)$$



This formula is the elegant way mathematics allows us to describe a **discrete (binned) probability** using the same language as **continuous distributions** (such as the bell curves in the Kalman Filter).

#### 1. What is the Dirac Delta Function $\delta(\mathbf{x}_t - \mathbf{x}^i)$?

Imagine the function $\delta(\cdot)$ as an **infinitely thin and tall pin** planted exactly at the point where its argument becomes zero:

* When $\mathbf{x}_t \neq \mathbf{x}^i$ (i.e., everywhere except at the center of cell $i$), the function equals **$0$**.
* When $\mathbf{x}_t = \mathbf{x}^i$ (exactly on the centroid $\mathbf{x}^i$ of cell $i$), the function has an **infinite peak** such that the total area under that pin is equal to **$1$**.

In practice, $\delta(\mathbf{x}_t - \mathbf{x}^i)$ acts as a **position switch**: it states *"all the mass of this term is concentrated exclusively at the fixed point $\mathbf{x}^i$"*.

#### 2. The Role of the Weight $w_t^i$

By multiplying the pin $\delta(\mathbf{x}_t - \mathbf{x}^i)$ by the number $w_t^i$ (the probability assigned to cell $i$), you are **scaling the importance of that pin**:

* If cell $i$ has probability $w_t^i = 0.60$, that peak concentrates $60\%$ of the robot's total probability.
* If cell $j$ has probability $w_t^j = 0$, the term $0 \cdot \delta(\mathbf{x}_t - \mathbf{x}^j)$ vanishes: the robot is definitely not there.

#### 3. The Summation $\sum_{i=1}^N$ (The Entire Grid)

The Duckiebot's map is divided into $N$ fixed cells, each identified by the coordinates of its center ($\mathbf{x}^1, \mathbf{x}^2, \dots, \mathbf{x}^N$).

By summing all $N$ terms, you obtain $bel(\mathbf{x}_t)$: instead of a smooth, continuous curve as in the Kalman Filter, the *belief* about the robot's position looks like a **bed of $N$ pins** planted at the center of each grid cell, each with a height proportional to the probability $w_t^i$ that the robot is located in that cell.

### Why Use This Notation Instead of a Simple List of Numbers?

On a computer (in Python/NumPy), $bel(\mathbf{x}_t)$ is simply stored as a matrix or an array of numbers $[w_t^1, w_t^2, \dots, w_t^N]$.

However, in the mathematical derivations of the **Bayes Filter** (where updating the state requires computing integrals $\int bel(\mathbf{x}) \, d\mathbf{x}$), writing $bel(\mathbf{x}_t)$ as a sum of Dirac deltas allows us to replace the continuous integral with a simple **finite sum** over all grid cells, making the formula compatible with general probability theory.

### The Goal in Duckietown: The Lane Filter

In the Duckiebot, the Histogram Filter is the "brain" that enables **autonomous lane following**.
Instead of estimating the absolute position in the world $(x, y)$, we estimate the **relative pose of the robot with respect to the road lane**, described by just two variables:

* **$d$ (lateral displacement):** The distance of the Duckiebot from the exact center of the right-hand lane (in meters).
* **$\phi$ (heading angle or orientation):** The angle of the Duckiebot relative to the direction of the lane (in radians).

The filter's API consists of three fundamental functions:

1. **`prior()`**: Initializes the probability grid $w_0$.
2. **`predict()`**: Propagates probability masses forward across the grid using wheel encoder data (odometry) and applies a Gaussian blur to model motion noise.
3. **`update()`**: Updates the probabilities using the road lines (white and yellow) detected by the camera through a geometric voting algorithm.



---

## Code Explanation: Setup and Libraries (Cell 1)

```python
%load_ext autoreload
%autoreload 2
%matplotlib inline

import cv2
import yaml
import matplotlib.pyplot as plt
import numpy as np
from scipy.ndimage import gaussian_filter
from scipy.stats import multivariate_normal
from math import floor, sqrt

print("Librerie importate con successo!")

```

* **Jupyter Magic Commands (`%load_ext`, `%autoreload`, `%matplotlib inline`):** Configure the notebook environment so that external Python modules are automatically reloaded if edited, and Matplotlib plots are rendered directly inside the notebook cells.
* **Computer Vision & Configuration (`cv2`, `yaml`):** `cv2` (OpenCV) is used for processing camera images and detecting road markings, while `yaml` is typically used to load Duckiebot calibration or filter parameters.
* **Numerical & Plotting Tools (`numpy`, `matplotlib.pyplot`):** `numpy` handles the 2D probability grid (the histogram matrix) and vectorized math operations, while `matplotlib` visualizes the belief distribution.
* **Statistical & Filtering Functions (`gaussian_filter`, `multivariate_normal`, `floor`, `sqrt`):**
* `gaussian_filter`: Applies spatial smoothing (blurring) to the 2D grid during the `predict()` step to account for odometry uncertainty.
* `multivariate_normal`: Generates 2D Gaussian distributions, commonly used to initialize the `prior()` belief around an expected starting pose.
* `floor` and `sqrt`: Standard math utilities used to discretize continuous coordinates into integer grid indices (bins) and compute Euclidean distances.





---

## Step 1: Defining the Lane State: $[d, \phi]$

In Duckietown's navigation system, the robot travels in the right-hand lane.
The state we want to estimate at every time step is the pair:

$$\mathbf{x} = \begin{bmatrix} d \\ \phi \end{bmatrix}$$

### Semantic Meaning of the Two Variables:

1. **$d$ (Distance from the lane center):**
* $d = 0$: The Duckiebot is perfectly centered in its lane.
* $d > 0$: The robot is shifted to the left (toward the solid yellow center line).
* $d < 0$: The robot is shifted to the right (toward the dashed white road-edge line).


2. **$\phi$ (Angle relative to the lane):**
* $\phi = 0$: The robot is pointing straight ahead, parallel to the road lines.
* $\phi > 0$: The robot is steering/angled to the left (counter-clockwise).
* $\phi < 0$: The robot is steering/angled to the right (clockwise).





---

## Code Explanation: State Visualization (Cell 2)

*(Note: The source code for the second code cell was cut off in the attached file because its base64 image output exceeded the text limit, though the cell output confirms it generates a Matplotlib figure.)*

* **Purpose:** In the Duckietown Lane Filter notebook, this cell immediately follows the $[d, \phi]$ state definition to initialize the discrete 2D histogram grid (defining the min/max bounds and bin resolution for lateral displacement $d$ and heading angle $\phi$) and plot a visual diagram illustrating the robot's pose within the lane. If you paste the remaining cells of the notebook, I can translate and explain those as well.

## Code Explanation: Loading the Camera Image (Cell 2)

```python
# Load the real image captured by the Duckiebot's camera
img_bgr = cv2.imread("../../assets/images/pic1.png")
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(8, 5))
plt.imshow(img_rgb)
plt.title("View from the Duckiebot Camera (pic1.png)", fontsize=13)
plt.axis("off")
plt.show()

```

* **`cv2.imread(...)` & `cv2.cvtColor(...)`:** Reads the sample road image (`pic1.png`) from disk in OpenCV's default BGR color format and converts it to standard RGB so Matplotlib renders the yellow and white lane markings accurately.
* **`plt.imshow(...)`:** Displays the raw camera perspective showing the right-hand lane before ground projection and line segmentation.

---

## Step 2: Configuration and 2D Grid Creation

The filter parameters (search bounds, road width, encoder resolution) are defined in the configuration file `assets/histogram_filter_config.yaml`.

We load these parameters and build our two-dimensional sampling grid (`np.mgrid`) for $d$ and $\phi$:

```python
# Load parameters from the YAML file
with open("../../assets/histogram_filter_config.yaml", "r") as stream:
    params = yaml.safe_load(stream)

hp = params["lane_filter_histogram_configuration"]

# Create the discrete grid (2D Meshgrid)
d_grid, phi_grid = np.mgrid[
    hp["d_min"] : hp["d_max"] : hp["delta_d"],
    hp["phi_min"] : hp["phi_max"] : hp["delta_phi"]
]

grid_spec = {
    "d": d_grid,
    "phi": phi_grid,
    "delta_d": hp["delta_d"],
    "delta_phi": hp["delta_phi"],
    "d_min": hp["d_min"],
    "d_max": hp["d_max"],
    "phi_min": hp["phi_min"],
    "phi_max": hp["phi_max"],
}

robot_spec = {
    "wheel_radius": hp["wheel_radius"],
    "encoder_resolution": hp["encoder_resolution"],
    "wheel_baseline": hp["wheel_baseline"],
}

road_spec = {
    "linewidth_white": hp["linewidth_white"],
    "linewidth_yellow": hp["linewidth_yellow"],
    "lanewidth": hp["lanewidth"],
}

cov_mask = [hp["sigma_d_mask"], hp["sigma_phi_mask"]]
mean_0 = [hp["mean_d_0"], hp["mean_phi_0"]]
cov_0 = [[hp["sigma_d_0"]**2, 0.0], [0.0, hp["sigma_phi_0"]**2]]

# Initialize the empty belief array with the same shape as the grid
belief = np.empty(d_grid.shape)
print(f"Histogram Grid Dimensions: {belief.shape} cells")
print(f"Resolution: delta_d = {hp['delta_d']} m, delta_phi = {hp['delta_phi']} rad")

```

* **`np.mgrid[...]`:** Generates two 2D coordinate matrices (`d_grid` and `phi_grid`) spanning from minimum to maximum bounds with step sizes `delta_d` and `delta_phi`. Every index `(i, j)` corresponds to a discrete state hypothesis $(d_i, \phi_j)$.
* **Specification Dictionaries (`grid_spec`, `robot_spec`, `road_spec`):** Group configuration constants cleanly so they can be passed into the filter functions (`prior`, `predict`, and `update`).

---

## Step 3: Belief Initialization — `histogram_prior`

At the very beginning, before the robot moves or reads any camera images, we must assign an initial probability $w_0$ to each cell in the grid. We assume the Duckiebot is placed roughly at the center of the lane ($d = 0$) and aligned straight ahead ($\phi = 0$).

We construct a 2D Gaussian distribution centered at $[0, 0]$ and evaluate its value at the centroid of each cell:


$$w_0^i = \frac{1}{2\pi \sigma_d \sigma_\phi} \exp\left( -\frac{d^2}{2\sigma_d^2} - \frac{\phi^2}{2\sigma_\phi^2} \right)$$

This section of the notebook (`03-histogram-filter`) defines **the starting state of the filter at time $t = 0$**, before the Duckiebot begins moving or capturing frames with its camera.

Unlike previous notebooks where you estimated $(x, y)$ coordinates on a generic map, here the filter performs **lane localization**, so the grid space is defined by two road-specific variables:

* **$d$ (lateral displacement):** How far the robot is shifted left or right relative to the lane centerline ($d = 0$ means dead center).
* **$\phi$ (heading angle):** How much the robot's nose is rotated relative to the direction of the road ($\phi = 0$ means perfectly straight and parallel to the lane markings).

### What the Equation Does Step-by-Step

Because we place the Duckiebot roughly in the center of the lane facing forward at startup, we want grid cells near $(d = 0, \phi = 0)$ to have a high initial weight $w_0^i$, while cells near the road edges or sharply angled should have weights close to zero. To achieve this bell-shaped profile, we use a **2D Gaussian distribution**:

$$w_0^i = \frac{1}{2\pi \sigma_d \sigma_\phi} \exp\left( -\frac{d^2}{2\sigma_d^2} - \frac{\phi^2}{2\sigma_\phi^2} \right)$$

Here is the role of each term evaluated at the center $(d, \phi)$ of the $i$-th grid cell:

1. **The scaling factor $\frac{1}{2\pi \sigma_d \sigma_\phi}$:** The normalization constant of a 2D Gaussian, ensuring the total volume under the continuous bell curve equals $1$. *(Practical note: since we evaluate this formula only on a finite grid of cells, in code we still divide all weights by their sum `weights /= np.sum(weights)` at the end to guarantee $\sum w_0^i = 1$).*
2. **The exponential $\exp(\dots)$:** Controls how rapidly probability drops off as you move away from the center $(0, 0)$:
* When $d = 0$ and $\phi = 0$, the exponent becomes $0$, and since $e^0 = 1$, that cell receives the maximum possible value.
* The larger $d$ or $\phi$ becomes (either positive or negative, since both are squared), the more negative the exponent becomes and $e^{-\text{number}}$ rapidly approaches **$0$**.


3. **The standard deviations $\sigma_d$ and $\sigma_\phi$:** Control how "wide" or "narrow" our initial confidence is:
* **Small $\sigma_d, \sigma_\phi$ values:** We are very confident the robot was placed straight in the center; almost all probability mass is concentrated in a few cells around $(0, 0)$.
* **Large $\sigma_d, \sigma_\phi$ values:** We are uncertain where the robot was placed; the bell curve flattens out and spreads weights more evenly across many cells.



### How It Translates into Notebook Code

In `histogram_prior`, we gather the center coordinates of every grid cell, evaluate this 2D Gaussian PDF to compute each cell's initial weight, and normalize the entire matrix so all cells sum to `1.0`:

```python
def histogram_prior(belief, grid_spec, mean_0, cov_0):
    """
    Initializes the histogram grid by sampling a 2D Gaussian density.
    """
    pos = np.empty(belief.shape + (2,))
    pos[:, :, 0] = grid_spec["d"]
    pos[:, :, 1] = grid_spec["phi"]
    
    RV = multivariate_normal(mean_0, cov_0)
    belief = RV.pdf(pos)
    # Normalize so the sum of all cells equals 1
    belief = belief / np.sum(belief)
    return belief

# Initialize and visualize the Prior
belief = histogram_prior(belief, grid_spec, mean_0, cov_0)

plt.figure(figsize=(7, 6))
plt.imshow(belief, origin="lower", cmap="jet", extent=[hp["phi_min"], hp["phi_max"], hp["d_min"], hp["d_max"]], aspect="auto")
plt.colorbar(label="Probability Mass")
plt.title("Initial Prior Belief bel(x_0)", fontsize=13)
plt.xlabel("Heading phi [rad]", fontsize=11)
plt.ylabel("Distance from lane center d [m]", fontsize=11)
plt.show()

```

---

## Step 4: Motion Prediction — `histogram_predict`

When the Duckiebot moves, optical wheel sensors (encoders) count how many "ticks" the left and right wheels have turned:

* $R$: wheel radius ($0.0318\text{ m}$).
* $\alpha = 2\pi / \text{resolution}$: angle per tick.
* Left wheel displacement: $d_L = R \cdot \alpha \cdot \text{ticks}_L$.
* Right wheel displacement: $d_R = R \cdot \alpha \cdot \text{ticks}_R$.

From **Differential Drive** kinematics:

1. **Forward travel along the robot's heading:**

$$d_A = \frac{d_L + d_R}{2}$$


2. **Angular rotation:**

$$w = \frac{d_R - d_L}{L_{\text{baseline}}}$$


3. **Lateral displacement relative to the lane:**
If the robot travels at orientation $\phi_t$, its lateral velocity component is:

$$v = d_A \sin(\phi_t)$$



### What Does `histogram_predict` Do on the Grid?

1. Takes the probability mass in each cell $(i, j)$ and computes its new position after motion: $d_{\text{new}} = d + v$, $\phi_{\text{new}} = \phi + w$.
2. Computes the new cell indices $(i_{\text{new}}, j_{\text{new}})$ and accumulates the probability mass there.
3. **Adding process noise:** In the real world, wheels slip! This uncertainty is modeled by applying a **2D Gaussian blur filter (`gaussian_filter`)** to the histogram, smoothly dispersing probability mass into neighboring cells—performing the exact same role as adding process covariance $+Q_t$ in the Kalman Filter.

This section implements the **Prediction step** of the Histogram Filter: it updates the probability map based solely on wheel encoder rotations, without looking at the camera yet.

#### 1. From Wheels to Lane Displacement (Kinematics)

* **Forward advancement ($d_A$):** The average displacement of the left and right wheels. If both advance by $10\text{ cm}$, the robot's center moves forward by $10\text{ cm}$.
* **Rotation ($w$):** The difference between the two wheels divided by the wheelbase ($L_{\text{baseline}}$, the distance between the wheels). If the right wheel turns more than the left, the robot turns left ($w > 0$); if both turn equally, it drives straight ($w = 0$).
* **Lateral shift ($v = d_A \sin(\phi_t)$):** For lane keeping, we only care about lateral offset ($d$) and angle ($\phi$), not longitudinal distance traveled down the road. If the robot is perfectly straight ($\phi_t = 0$), $\sin(0) = 0$ and moving forward does not change its distance from the lane center ($v = 0$). If angled toward a lane boundary ($\phi_t \neq 0$), advancing by $d_A$ shifts it laterally by $d_A \sin(\phi_t)$.

#### 2. How Probability Moves Across the Grid

1. **Shifting hypotheses:** For every cell $(i, j)$ on the grid, we compute where the robot would end up starting from $(d, \phi)$ after applying the measured motion:

$$d_{\text{new}} = d + v, \qquad \phi_{\text{new}} = \phi + w$$


2. **Re-binning:** Continuous coordinates $(d_{\text{new}}, \phi_{\text{new}})$ rarely land dead-center on a grid cell, so they are floored to the nearest cell indices $(i_{\text{new}}, j_{\text{new}})$. Because multiple starting cells may land in the same destination cell, their probabilities are **summed** into the new grid.

#### 3. Gaussian Blur (`gaussian_filter`) = Process Noise

Without smoothing, the histogram would shift rigidly as if wheels never slipped. Applying `scipy.ndimage.gaussian_filter` takes a fraction of probability from each active cell and spreads it to adjacent neighbors, lowering peak sharpness and widening the uncertainty blob—mirroring $+Q_t$ in a Kalman Filter.

```python
def get_estimate(belief, grid_spec):
    """Computes the expected value (weighted mean) of state d and phi."""
    d_est = np.sum(belief * grid_spec["d"])
    phi_est = np.sum(belief * grid_spec["phi"])
    return d_est, phi_est


def histogram_predict(belief, left_encoder_ticks, right_encoder_ticks, grid_spec, robot_spec, cov_mask):
    """
    Propagates the belief across the grid using wheel encoder data.
    """
    belief_in = belief
    
    # 1. Compute wheel displacements
    R = robot_spec["wheel_radius"]
    alpha = 2 * np.pi / robot_spec["encoder_resolution"]
    d_left = R * alpha * left_encoder_ticks
    d_right = R * alpha * right_encoder_ticks
    
    d_A = (d_left + d_right) / 2.0
    w = (d_right - d_left) / robot_spec["wheel_baseline"]
    
    _, phi_curr = get_estimate(belief, grid_spec)
    v = d_A * np.sin(phi_curr)
    
    # New centroid coordinates
    d_t = grid_spec["d"] + v
    phi_t = grid_spec["phi"] + w
    
    p_belief = np.zeros(belief.shape)
    
    # 2. Re-assign probability mass to the corresponding new cells
    for i in range(belief.shape[0]):
        for j in range(belief.shape[1]):
            if belief[i, j] > 0:
                # Discard mass if it moves outside grid boundaries
                if (
                    d_t[i, j] > grid_spec["d_max"]
                    or d_t[i, j] < grid_spec["d_min"]
                    or phi_t[i, j] < grid_spec["phi_min"]
                    or phi_t[i, j] > grid_spec["phi_max"]
                ):
                    continue
                
                i_new = int(floor((d_t[i, j] - grid_spec["d_min"]) / grid_spec["delta_d"]))
                j_new = int(floor((phi_t[i, j] - grid_spec["phi_min"]) / grid_spec["delta_phi"]))
                
                if 0 <= i_new < belief.shape[0] and 0 <= j_new < belief.shape[1]:
                    p_belief[i_new, j_new] += belief[i, j]
                    
    # 3. Model process noise via Gaussian filter (uncertainty blur)
    s_belief = np.zeros(belief.shape)
    gaussian_filter(p_belief, cov_mask, output=s_belief, mode="constant")
    
    if np.sum(s_belief) == 0:
        return belief_in
        
    # Normalize so total sum remains 1
    belief = s_belief / np.sum(s_belief)
    return belief

# Simulate a motion step: 10 ticks left and 20 ticks right (the robot turns left)
belief_pred = histogram_predict(belief, left_encoder_ticks=10, right_encoder_ticks=20, 
                                grid_spec=grid_spec, robot_spec=robot_spec, cov_mask=cov_mask)

plt.figure(figsize=(7, 6))
plt.imshow(belief_pred, origin="lower", cmap="jet", extent=[hp["phi_min"], hp["phi_max"], hp["d_min"], hp["d_max"]], aspect="auto")
plt.colorbar(label="Probability Mass")
plt.title("Predicted Belief after Motion (bar_bel)", fontsize=13)
plt.xlabel("Heading phi [rad]", fontsize=11)
plt.ylabel("Distance from lane center d [m]", fontsize=11)
plt.show()

```

---

## Step 5: The Visual Measurement Model — Segments, Projection, and Votes

Now comes the most fascinating part of the Duckiebot's perception pipeline:

1. **Segment Detection:** The camera detects line segments belonging to road markings (white on the right, yellow on the left).
2. **Ground Projection:** Using a homography matrix and camera calibration, each 2D image segment is projected onto the 3D ground plane in meters.
3. **The Geometric Voting Algorithm (`generate_vote`):**
* Suppose a white segment is observed on the ground with endpoints $p_1$ and $p_2$.
* We compute the unit tangent vector $\hat{t}$ and unit normal vector $\hat{n}$:

$$\hat{t} = \frac{p_2 - p_1}{\Vert{}p_2 - p_1\Vert{}}, \quad \hat{n} = \begin{bmatrix} -\hat{t}_y \\ \hat{t}_x \end{bmatrix}$$


* We compute the perpendicular distance $d_i$ and heading angle $\phi_i = \arcsin(\hat{t}_y)$.
* We offset $d_i$ using the known geometry of the road:
* The **white** line lies on the right edge (at offset $-W_{\text{lane}} / 2$).
* The **yellow** line lies on the centerline (at offset $+W_{\text{lane}} / 2$).


* In this way, **every single detected segment casts a vote** for what pose $[d, \phi]$ the robot must currently have!



```python
# Data structures for line segments
class SegmentColor:
    WHITE = 0
    YELLOW = 1
    RED = 2

class SegmentPoint:
    def __init__(self, x, y):
        self.x = x
        self.y = y
    def as_array(self):
        return np.array([self.x, self.y])

class Segment:
    def __init__(self, p1, p2, color):
        self.points = [SegmentPoint(p1[0], p1[1]), SegmentPoint(p2[0], p2[1])]
        self.color = color


def prepare_segments(segments):
    """Filters out red stop-line segments or segments located behind the robot."""
    filtered = []
    for seg in segments:
        if seg.color not in [SegmentColor.WHITE, SegmentColor.YELLOW]:
            continue
        # Filter out segments behind the vehicle (x < 0)
        if seg.points[0].x < 0 or seg.points[1].x < 0:
            continue
        filtered.append(seg)
    return filtered


def generate_vote(segment, road_spec):
    """
    Given a ground-projected road segment, computes the (d, phi) pose estimate
    voted by that segment based on standard lane geometry.
    """
    p1 = segment.points[0].as_array()
    p2 = segment.points[1].as_array()
    diff = p2 - p1
    norm = np.linalg.norm(diff)
    if norm == 0:
        return 0.0, 0.0
        
    t_hat = diff / norm
    n_hat = np.array([-t_hat[1], t_hat[0]])
    
    d1 = np.inner(n_hat, p1)
    d2 = np.inner(n_hat, p2)
    d_i = (d1 + d2) / 2.0
    phi_i = np.arcsin(np.clip(t_hat[1], -1.0, 1.0))
    
    if segment.color == SegmentColor.WHITE:  # White right-edge line
        if p1[0] > p2[0]:
            d_i -= road_spec["linewidth_white"]
        else:
            d_i = -d_i
            phi_i = -phi_i
        d_i -= road_spec["lanewidth"] / 2.0
        
    elif segment.color == SegmentColor.YELLOW:  # Yellow center line (left)
        if p2[0] > p1[0]:
            d_i -= road_spec["linewidth_yellow"]
            phi_i = -phi_i
        else:
            d_i = -d_i
        d_i = road_spec["lanewidth"] / 2.0 - d_i
        
    return d_i, phi_i

```

---

## Step 6: Building the Vote Histogram — `generate_measurement_likelihood`

All detected segments cast a vote for a cell $(d_i, \phi_i)$ on the grid:

* We initialize a matrix of zeros matching the grid dimensions.
* For each segment, we compute $(d_i, \phi_i)$.
* If the vote falls within grid bounds, we increment the corresponding cell count by $+1$.
* We normalize the matrix so its entries sum to $1$. This gives us the **Measurement Likelihood** $p(\mathbf{z}_t \mid \mathbf{x}^i)$!

### 1. Step-by-Step Construction of the Vote Matrix

* **Accumulation grid initialization:** A NumPy zero matrix is created with the exact same dimensions and ranges as the robot's belief grid.
* **Vote collection and outlier rejection:** For every detected segment $s^i$, its vote $(d^i, \phi^i)$ is computed via `generate_vote`. Votes outside the physical bounds of the grid (e.g., caused by floor glare or off-road false detections) are discarded.
* **Binning and counting:** Continuous coordinates $(d^i, \phi^i)$ are converted into integer cell indices and incremented by $+1$. Cells agreed upon by many real road segments accumulate high counts, while isolated noise votes stay near 1 or 2.
* **Normalization:** Dividing every cell by the total number of valid votes ensures the matrix sums to $1$:

$$p(\mathbf{z}_t \mid \mathbf{x}^i) = \frac{\text{votes in cell } i}{\text{total valid votes}}$$



*(Practical safeguard: if no segments are visible in a frame and total votes equal $0$, the function returns `None` so the update step leaves the odometry prediction unchanged).*

### 2. Mathematical Meaning of Measurement Likelihood $p(\mathbf{z}_t \mid \mathbf{x}^i)$

In Bayes' rule, $p(\mathbf{z}_t \mid \mathbf{x}^i)$ reads: **"the probability of observing the current measurement $\mathbf{z}_t$ (the segments seen in the image), assuming the robot is located in cell $\mathbf{x}^i$."**

* It represents **only** what the camera sees right now, ignoring prior motion.
* In the final `update` step, multiplying $p(\mathbf{z}_t \mid \mathbf{x}^i)$ element-by-element with the odometry prediction $\overline{bel}(\mathbf{x}^i)$ suppresses visual outliers and sharpens the pose estimate.

```python
def generate_measurement_likelihood(segments, road_spec, grid_spec):
    """
    Generates the 2D measurement likelihood histogram by accumulating votes from all segments.
    """
    measurement_likelihood = np.zeros(grid_spec["d"].shape)
    
    for seg in segments:
        d_i, phi_i = generate_vote(seg, road_spec)
        
        # Ignore votes outside histogram boundaries
        if (
            d_i > grid_spec["d_max"]
            or d_i < grid_spec["d_min"]
            or phi_i < grid_spec["phi_min"]
            or phi_i > grid_spec["phi_max"]
        ):
            continue
            
        i = int(floor((d_i - grid_spec["d_min"]) / grid_spec["delta_d"]))
        j = int(floor((phi_i - grid_spec["phi_min"]) / grid_spec["delta_phi"]))
        
        if 0 <= i < measurement_likelihood.shape[0] and 0 <= j < measurement_likelihood.shape[1]:
            measurement_likelihood[i, j] += 1.0
            
    if np.sum(measurement_likelihood) == 0:
        return None
        
    measurement_likelihood /= np.sum(measurement_likelihood)
    return measurement_likelihood

```

---

## Step 7: Bayesian Update — `histogram_update`

The Bayesian update on a discrete grid is remarkably clean:


$$bel(\mathbf{x}_t) = \frac{\bar{w}_t^i \cdot p(\mathbf{z}_t \mid \mathbf{x}^i)}{\sum_j \bar{w}_t^j \cdot p(\mathbf{z}_t \mid \mathbf{x}^j)}$$

In Python and NumPy, this is simply an **element-wise multiplication** (`np.multiply`, also known as the Hadamard product) between the predicted belief matrix and the camera likelihood matrix, followed by dividing by the total sum!

### 1. The Numerator: $\bar{w}_t^i \cdot p(\mathbf{z}_t \mid \mathbf{x}^i)$ (Logical "AND")

* **$\bar{w}_t^i$ (Prediction / Prior):** Output from `histogram_predict`, encoding where wheel odometry expects the robot to be.
* **$p(\mathbf{z}_t \mid \mathbf{x}^i)$ (Measurement Likelihood):** Output from `generate_measurement_likelihood`, encoding where camera line detections place the robot.

Multiplying the two acts like a **logical AND**: for a cell to retain high probability, **both odometry and vision must agree**. Visual reflections where the robot couldn't physically be are zeroed out, while wide odometry uncertainty blobs are trimmed down to the sharp visual intersection.

### 2. The Denominator: $\sum_j \bar{w}_t^j \cdot p(\mathbf{z}_t \mid \mathbf{x}^j)$ (Normalization)

Multiplying two grids of probabilities ($< 1$) yields small numbers whose sum is no longer $1$. Dividing by the sum of all cells rescales the posterior grid back into a valid probability distribution summing to $1.0$.

### 3. Edge-Case Handling & Extracting the Final Estimate

* **Zero-mass safeguard (`if total_mass == 0`):** If odometry and camera completely disagree (e.g., after a bump or "kidnapped robot" event) so no active cells overlap, the product is all zeros. Falling back to `measurement_likelihood` (or keeping `prior`) prevents division-by-zero `NaN` errors.
* **Extracting $(d_t, \phi_t)$:** You can extract either the **MAP (Maximum A Posteriori)** peak cell (`np.argmax`) or the **weighted expected value** (`get_estimate`), which provides a smooth, continuous estimate without grid discretization jumps.

```python
def histogram_update(belief, segments, road_spec, grid_spec):
    """
    Updates the belief by multiplying the predicted prior by the measurement likelihood.
    """
    segments_filtered = prepare_segments(segments)
    measurement_likelihood = generate_measurement_likelihood(segments_filtered, road_spec, grid_spec)
    
    if measurement_likelihood is not None:
        # Bayesian fusion: element-wise multiplication (Hadamard product)
        belief = np.multiply(belief, measurement_likelihood)
        
        # Normalization
        total_mass = np.sum(belief)
        if total_mass == 0:
            belief = measurement_likelihood
        else:
            belief /= total_mass
            
    return measurement_likelihood, belief

```

---

## Step 8: Full Cycle Test and Final Visualization

We create a synthetic set of detected road segments consistent with our lane (white segments on the right, yellow segments on the left) and run `histogram_update` starting from our predicted belief:

* Visualize the **Odometry Prediction** ($\overline{bel}$).
* Visualize the **Measurement Likelihood** (what the camera sees).
* Visualize the **Final Updated Belief** (the optimal probabilistic fusion).

```python
# Create realistic ground-projected test segments
# 1. White line on the right (x between 0.2 and 0.8 m ahead of the robot, y around -0.11 m)
seg_white_1 = Segment([0.25, -0.11], [0.45, -0.11], SegmentColor.WHITE)
seg_white_2 = Segment([0.55, -0.11], [0.75, -0.11], SegmentColor.WHITE)

# 2. Yellow line on the left (x between 0.2 and 0.7 m ahead, y around +0.12 m)
seg_yellow_1 = Segment([0.20, 0.12], [0.40, 0.12], SegmentColor.YELLOW)
seg_yellow_2 = Segment([0.45, 0.12], [0.65, 0.12], SegmentColor.YELLOW)

test_segments = [seg_white_1, seg_white_2, seg_yellow_1, seg_yellow_2]

# Run the update step
meas_like, belief_final = histogram_update(belief_pred, test_segments, road_spec, grid_spec)

# Plot results side by side
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Odometry Prediction
im1 = axes[0].imshow(belief_pred, origin="lower", cmap="jet", extent=[hp["phi_min"], hp["phi_max"], hp["d_min"], hp["d_max"]], aspect="auto")
axes[0].set_title("1. Odometry Prediction (bar_bel)", fontsize=13)
axes[0].set_xlabel("phi [rad]")
axes[0].set_ylabel("d [m]")
plt.colorbar(im1, ax=axes[0])

# 2. Camera Likelihood (Votes)
im2 = axes[1].imshow(meas_like, origin="lower", cmap="jet", extent=[hp["phi_min"], hp["phi_max"], hp["d_min"], hp["d_max"]], aspect="auto")
axes[1].set_title("2. Camera Segment Votes (Likelihood)", fontsize=13)
axes[1].set_xlabel("phi [rad]")
axes[1].set_ylabel("d [m]")
plt.colorbar(im2, ax=axes[1])

# 3. Final Fused Posterior
im3 = axes[2].imshow(belief_final, origin="lower", cmap="jet", extent=[hp["phi_min"], hp["phi_max"], hp["d_min"], hp["d_max"]], aspect="auto")
axes[2].set_title("3. Final Updated Belief bel(x_t)", fontsize=13)
axes[2].set_xlabel("phi [rad]")
axes[2].set_ylabel("d [m]")
plt.colorbar(im3, ax=axes[2])

plt.tight_layout()
plt.show()

d_final, phi_final = get_estimate(belief_final, grid_spec)
print(f"Final Optimal Lane Pose Estimate:")
print(f" -> Distance from center (d):   {d_final:.4f} meters")
print(f" -> Heading angle (phi):        {phi_final:.4f} radians ({np.degrees(phi_final):.2f} degrees)")

```

---

## Summary Table of the Histogram Lane Filter

| Component | Physical Meaning | How It Is Computed in Python |
| --- | --- | --- |
| **$d$** | Lateral distance from lane center | Vertical axis of the grid ($-0.3\text{ m} \dots +0.3\text{ m}$) |
| **$\phi$** | Heading angle relative to the lane | Horizontal axis of the grid ($-1.5\text{ rad} \dots +1.5\text{ rad}$) |
| **`histogram_prior`** | Initial prior knowledge | Sampling a 2D multivariate normal centered at $(0, 0)$ |
| **`histogram_predict`** | Robot motion from wheel encoders | Encoder tick integration + convolution with `gaussian_filter` |
| **`generate_vote`** | Pose estimate from a white/yellow line | Vector geometry: unit normal, tangent, and lane width offset |
| **`generate_measurement_likelihood`** | Camera vote map | 2D histogram accumulating $+1$ per valid road segment |
| **`histogram_update`** | Optimal fusion of odometry & vision | Element-wise product (`np.multiply`) + normalization |

---

### Next Step: The Extended Kalman Filter (EKF)

In the next and final notebook, we will tackle the pinnacle of robotic localization: **[suspicious link removed]**, where we use landmarks (AprilTags) and Jacobian matrices to linearize the Duckiebot's navigation in real time!

# EXTRA NOTES

### Prediction Step (`predict()`)

The `predict()` function propagates forward the belief weights based on the motion model and the control, $u_t$. This amounts to propagating the centroids of each of the cells forward and then adding all of the weight up that lands in each bin.

$$\overline{bel}(x_t) =  \sum_{i=1}^N \sum_{j=1}^N w_{t-1}^j p(x^i \mid x^j, u_t) \delta(x_t - x^i)$$

In our case we will be using the odometry as a proxy for the control input so that we may use a simple kinematic model of the robot.

#### Explanation:

This formula is the exact mathematical translation of the English statement above: **take the starting probabilities, shift them forward according to the motion performed, and sum up everything that lands in each individual cell**.

To understand it thoroughly, let's break it down piece by piece and then see how the two summations work together.

##### 1. The Individual Ingredients of the Formula

* **$\overline{bel}(x_t)$ (Predicted Belief):** This is the new probability grid at time $t$, immediately after the motion occurs but **before** looking through the camera. The bar on top ($\overline{bel}$) specifically distinguishes it from the final, corrected *belief*.
* **$N$:** The total number of cells (bins) in your grid.
* **$j$ and $i$ (Cell Indices):**
* $j$ denotes a **starting cell** (at time $t-1$).
* $i$ denotes a **destination cell** (at time $t$).


* **$w_{t-1}^j$ (Starting Weight):** The probability that the robot was located in cell $j$ before moving.
* **$p(x^i \mid x^j, u_t)$ (Motion / Transition Model):** The probability of ending up at the centroid of cell $i$ ($x^i$), given that the robot started at the centroid of cell $j$ ($x^j$) and executed the motion command $u_t$ (e.g., *"turn the wheels by X degrees"*). Because wheels can slip, this value is almost never strictly $1$ or $0$; instead, it distributes probability across a primary destination cell and slightly into neighboring cells (the Gaussian blur).
* **$\delta(x_t - x^i)$ (Dirac Delta):** As seen earlier, this is the mathematical "pin" stating: *"all the probability we just computed for cell $i$ is placed at its exact center point $x^i$"*.

##### 2. How the Mechanism Works (The Double Summation)

The core of the formula lies in the two summations $\sum_{i=1}^N \sum_{j=1}^N$. Let's read them from the inside out:

* **Step A: The inner summation $\sum_{j=1}^N w_{t-1}^j p(x^i \mid x^j, u_t)$**
Let's fix a specific destination cell—for example, **cell $i = 5$**. We ask: *"How much total probability lands in cell 5 after the movement?"*
Some probability might arrive in cell 5 from any starting cell $j$:
1. Take starting cell $j=1$: it had weight $w_{t-1}^1$. Multiply it by the probability of moving from $1$ to $5$ given control $u_t$.
2. Do the same starting from $j=2$, $j=3$, and so on across all $N$ cells.
3. **Sum all these contributions** ($\sum_{j=1}^N$). The result is the **new predicted weight** $\overline{w}_t^5$ of cell 5.
*(In statistics, this step is known as the Law of Total Probability).*


* **Step B: The outer summation $\sum_{i=1}^N (...) \delta(x_t - x^i)$**
Once you have computed the new weight for cell 5, you repeat the exact same calculation for every other destination cell ($i = 1, 2, \dots, N$) and place each weight onto its respective centroid via $\delta(x_t - x^i)$ to reconstruct the entire distribution across the grid.

##### 3. A Practical Example with 3 Cells

Imagine a single row of just 3 cells: **[1]**, **[2]**, **[3]**.

Before moving ($t-1$), the robot believes it is in cell 1 with probability $0.8$ ($w_{t-1}^1 = 0.8$) and in cell 2 with probability $0.2$ ($w_{t-1}^2 = 0.2$). Cell 3 has probability $0$.

Now the robot moves forward by one cell ($u_t$). Due to wheel uncertainty, when commanded to move 1 step forward, there is a **70%** chance it actually advances by 1 cell, and a **30%** chance it advances by 2 cells.

Let's calculate how much probability lands in **destination cell $i = 3$** (the inner summation):

* **Contribution from $j=1$:** The robot starts at 1 (weight $0.8$) and advances 2 cells to land in 3 (probability $0.3$). Contribution: $0.8 \times 0.3 = \mathbf{0.24}$.
* **Contribution from $j=2$:** The robot starts at 2 (weight $0.2$) and advances 1 cell to land in 3 (probability $0.7$). Contribution: $0.2 \times 0.7 = \mathbf{0.14}$.
* **Contribution from $j=3$:** It started with weight $0$, so its contribution is $\mathbf{0}$.

Summing everything that landed in cell 3:


$$\overline{w}_t^3 = 0.24 + 0.14 + 0 = \mathbf{0.38}$$

In the notebook's code, this double loop corresponds directly to shifting the starting centroids forward through the robot's kinematic model (adding the motion $u_t$), applying the Gaussian filter to spread uncertainty, and summing the values that fall within the boundaries of each matrix cell.

---

### Update Step (`update()`)

Finally, the `update()` function takes a measurement and uses it to update the weights of the histogram bins based on the incoming measurement, $z_t$. This is achieved by multiplying the weight in each bin by the likelihood that the measurement was generated by the state corresponding to the centroid of that bin:

$$bel(x_t) = \sum_{i=1}^N \frac{\overline{w}^i_t p(z_t \mid x^i)}{\displaystyle\sum_{j=1}^N \overline{w}^j_t p(z_t \mid x^j)}\delta(x_t - x^i)$$

In this notebook we will proceed by loading one image and using the line detection and ground projection algorithms (we can consider them as a black box here) to detect all of the white and yellow line segments. Each line segment will contribute a vote in the measurement likelihood.

#### Explanation:

This formula is the direct application of **Bayes' Rule** on every individual cell of the grid: it takes the "blurred" probabilities obtained from motion ($\overline{w}_t^i$) and updates them by rewarding the cells that agree with what the camera sees ($z_t$).

Let's break it down piece by piece to see how it works.

##### 1. The Individual Ingredients of the Formula

* **$bel(x_t)$ (Final / Posterior Belief):** The definitive probability distribution at time $t$, after combining both motion and camera observation.
* **$\overline{w}_t^i$ (Predicted Weight):** The weight of cell $i$ fresh out of the `predict()` function (the one with the bar on top). It represents how strongly we believed the robot was in cell $i$ *before* looking at the camera image.
* **$p(z_t \mid x^i)$ (Measurement Likelihood):** The heart of the update step. It indicates how likely it is to observe the road segments $z_t$ at the positions seen by the camera, **assuming the robot is located at the exact centroid $x^i$**:
* If, from cell $x^i$, the yellow line should appear right where the camera sees it, this value will be **high**.
* Conversely, if from cell $x^i$ the yellow line would be on the opposite side of the road, this value will be **close to zero**.


* **$\delta(x_t - x^i)$ (Dirac Delta):** Just like in the previous formula, this simply "plants" the newly computed weight at the exact center $x^i$ of cell $i$.

##### 2. The Fraction: Why Multiply and Then Divide?

For each cell $i$, the new definitive weight $w_t^i$ is given by the fraction:

$$w_t^i = \frac{\overline{w}^i_t p(z_t \mid x^i)}{\sum_{j=1}^N \overline{w}^j_t p(z_t \mid x^j)}$$

1. **The numerator ($\overline{w}^i_t \cdot p(z_t \mid x^i)$):** Cross-references the two sources of information. A cell achieves a high score only if **both** factors are high: it must already be plausible after motion ($\overline{w}_t^i$ high) **and** consistent with what the camera sees ($p(z_t \mid x^i)$ high). If either factor is zero, the product drops to zero.
2. **The denominator ($\sum_{j=1}^N \overline{w}^j_t p(z_t \mid x^j)$):** This is the **normalization** factor. When you multiply numbers smaller than $1$ together, the sum of all cell weights becomes much smaller than $1$. Dividing each numerator by the sum of all numerators across the grid rescales everything so that the total probability sums back to exactly **$1$** (100%).

##### 3. How Does White and Yellow Segment "Voting" Fit In?

The paragraph below the formula explains how the likelihood grid $p(z_t \mid x^i)$ is constructed in practice:

1. The camera captures an image, and a computer vision algorithm detects dozens of small **white** (right edge) and **yellow** (centerline) segments, projecting them onto the floor ahead of the robot.
2. Knowing the standard width of a Duckietown lane, **each individual segment** allows an inverse geometric calculation: *"If I see a yellow segment 15 cm to my left at this angle, then I (the robot) must be located in cell $(d, \phi)$."*
3. Each detected segment therefore casts a **"vote"** into the corresponding $(d, \phi)$ cell. The more segments agree on the same pose, the more votes that cell accumulates and the higher its $p(z_t \mid x^i)$ value becomes.

##### 4. A Practical Example with 3 Cells

Let's return to our 3 cells **[1]**, **[2]**, **[3]** with the weights just produced by `predict()`:

* $\overline{w}_t^1 = 0.20$
* $\overline{w}_t^2 = 0.42$
* $\overline{w}_t^3 = 0.38$

Now the camera looks at the road ($z_t$) and computes the likelihood $p(z_t \mid x^i)$ based on the votes from the white and yellow lines:

* For cell 1: $p(z_t \mid x^1) = 0.1$ (poorly consistent with the observed lines)
* For cell 2: $p(z_t \mid x^2) = 0.8$ (highly consistent!)
* For cell 3: $p(z_t \mid x^3) = 0.1$ (poorly consistent)

**Step 1: Compute the numerators (predicted weight $\times$ likelihood)**

* Cell 1: $0.20 \times 0.1 = \mathbf{0.020}$
* Cell 2: $0.42 \times 0.8 = \mathbf{0.336}$
* Cell 3: $0.38 \times 0.1 = \mathbf{0.038}$

**Step 2: Compute the denominator (sum of all numerators)**


$$\text{Sum} = 0.020 + 0.336 + 0.038 = \mathbf{0.394}$$

**Step 3: Divide (Normalization)**

* $w_t^1 = \frac{0.020}{0.394} \approx \mathbf{0.05}$ (5%)
* $w_t^2 = \frac{0.336}{0.394} \approx \mathbf{0.85}$ (85%)
* $w_t^3 = \frac{0.038}{0.394} \approx \mathbf{0.10}$ (10%)

The probability is now concentrated at **85%** in cell 2, eliminating the uncertainty that had accumulated during the motion step.